# Imports y rutas

Para instalar cv2 hay que hacerlo con `pip install opencv-python`.<br>
Para instalar tensorflow hay que hacerlo con `pip install tensorflow`.

In [1]:
# 1. Imports y rutas

import os
import cv2
import xml.etree.ElementTree as ET
from tqdm import tqdm

TRAIN_DIR = "./../../ImagesData/Playing Cards Images - Object Detection Dataset/train"
TEST_DIR  = "./../../ImagesData/Playing Cards Images - Object Detection Dataset/test"

OUT_TRAIN = "dataset/train"
OUT_TEST  = "dataset/test"

os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)


In [2]:
# 2. Función para procesar una carpeta (leer XML y recortar)

def process_folder(src_dir, out_dir):
    for file in tqdm(os.listdir(src_dir)):
        if not file.endswith(".xml"):
            continue

        xml_path = os.path.join(src_dir, file)
        tree = ET.parse(xml_path)
        root = tree.getroot()

        filename = root.find("filename").text
        img_path = os.path.join(src_dir, filename)

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)

        obj = root.find("object")
        label = obj.find("name").text.replace(" ", "_")

        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        crop = img[ymin:ymax, xmin:xmax]

        class_dir = os.path.join(out_dir, label)
        os.makedirs(class_dir, exist_ok=True)

        cv2.imwrite(os.path.join(class_dir, filename), crop)


In [3]:
# 3. Crear dataset de entrenamiento y test

process_folder(TRAIN_DIR, OUT_TRAIN)
process_folder(TEST_DIR, OUT_TEST)


100%|██████████| 196/196 [00:00<00:00, 276.57it/s]


In [4]:
# 4. Cargar datos con Keras

import tensorflow as tf
# To use tensorflow we can't use python 3.13, we create a new enviroment with an older version
# conda create -n tf_env python=3.10
# conda activate tf_env
# conda install -c conda-forge tensorflow

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    OUT_TRAIN,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_gen = test_datagen.flow_from_directory(
    OUT_TEST,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_gen.num_classes
print("Clases:", train_gen.class_indices)


Found 539 images belonging to 52 classes.
Found 98 images belonging to 50 classes.
Clases: {'ace_of_clubs': 0, 'ace_of_diamonds': 1, 'ace_of_hearts': 2, 'ace_of_spades': 3, 'eight_of_diamonds': 4, 'eight_of_hearts': 5, 'eight_of_spades': 6, 'eigth_of_clubs': 7, 'five_of_clubs': 8, 'five_of_diamonds': 9, 'five_of_hearts': 10, 'five_of_spades': 11, 'four_of_clubs': 12, 'four_of_diamonds': 13, 'four_of_hearts': 14, 'four_of_spades': 15, 'jack_of_clubs': 16, 'jack_of_diamonds': 17, 'jack_of_hearts': 18, 'jack_of_spades': 19, 'king_of_clubs': 20, 'king_of_diamonds': 21, 'king_of_hearts': 22, 'king_of_spades': 23, 'nine_of_clubs': 24, 'nine_of_diamonds': 25, 'nine_of_hearts': 26, 'nine_of_spades': 27, 'queen_of_clubs': 28, 'queen_of_diamonds': 29, 'queen_of_hearts': 30, 'queen_of_spades': 31, 'seven_of_clubs': 32, 'seven_of_diamonds': 33, 'seven_of_hearts': 34, 'seven_of_seven': 35, 'seven_of_spades': 36, 'six_of_diamonds': 37, 'six_of_hearts': 38, 'six_of_spades': 39, 'ten_of_clubs': 40, 't

In [5]:
# 5. Modelo CNN

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(base_model.input, output)

model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,125,620 (92.03 MB)

 Trainable params: 537,908 (2.05 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [6]:
# 6. Entrenamiento

history = model.fit(
    train_gen,
    epochs=15
)


Epoch 1/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 31s 718ms/step - accuracy: 0.0167 - loss: 4.2134
Epoch 2/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 636ms/step - accuracy: 0.0390 - loss: 3.9435
Epoch 3/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 637ms/step - accuracy: 0.0649 - loss: 3.8532
Epoch 4/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 645ms/step - accuracy: 0.0724 - loss: 3.7899
Epoch 5/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 622ms/step - accuracy: 0.0816 - loss: 3.7374
Epoch 6/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 617ms/step - accuracy: 0.0761 - loss: 3.6780
Epoch 7/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 621ms/step - accuracy: 0.1020 - loss: 3.6357
Epoch 8/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 611ms/step - accuracy: 0.0983 - loss: 3.5914
Epoch 9/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 604ms/step - accuracy: 0.1150 - loss: 3.5584
Epoch 10/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 610ms/step - accuracy: 0.1169 - loss: 3.5130
Epoch 11/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 643ms/step - accuracy: 0.1243 - loss: 3.4919
Epoch 12/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 22

In [7]:
# 7. Evaluación en test

test_loss, test_acc = model.evaluate(test_gen)
print("Accuracy en test:", test_acc)


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 50), output.shape=(None, 52)

In [ ]:
# Predicción de una carta correcta

import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "Datos cartas/test/201.jpg"

img = image.load_img(img_path, target_size=IMG_SIZE)
x = image.img_to_array(img) / 255.0
x = np.expand_dims(x, axis=0)

pred = model.predict(x)
pred_idx = np.argmax(pred)

labels = {v: k for k, v in train_gen.class_indices.items()}
print("Carta predicha:", labels[pred_idx])
